In [1]:
# Necessary imports
import os
import pandas as pd
import numpy as np

# Repository root: holds config.py (paths and per-cancer settings) and driver_tiers.py
import sys
repo_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
import config

# CANCER CONFIGURATION: the cancer and the data root come from config.py
CANCER     = config.CANCER

# folder holding this cancer's run
base_dir   = config.BASE_DIR

# this cancer's own folder: holds <prefix>_data, <prefix>_notebooks and <prefix>_output
cancer_dir = os.path.join(base_dir, CANCER.lower())

# this cancer's input data (expression files, downloaded clinical / validation cohorts)
data_dir   = os.path.join(cancer_dir, f"{CANCER.lower()}_data")
prefix     = CANCER.lower()
output_dir = os.path.join(cancer_dir, f"{prefix}_output")

# survival endpoint: "OS" (overall survival), "DSS" (disease-specific), "DFS" (disease-free) or "PFS" (progression-free)
# THCA (16 deaths in 499 patients) and PRAD have too few deaths to fit or test a survival model, so they use progression-free survival
# PFS and DSS are only in the PanCancer Atlas clinical files (Liu et al. 2018, Cell), not in the legacy TCGA study
# If the endpoint is changed after 05 has run, delete guan_scores.csv and regulon_cox_results.csv in the output folder first
ENDPOINTS  = config.ENDPOINTS
ENDPOINT   = ENDPOINTS.get(CANCER, "OS")

# cBioPortal TCGA study for each cancer ; colon cancer is combined with rectal in cBioPortal
CANCER_SETTINGS = config.CBIOPORTAL_STUDIES

CBIO_STUDY = CANCER_SETTINGS[CANCER]

# clinical file from the cBioPortal study download: nested (brca_tcga/brca_tcga/) or directly inside the study folder
clin_path = os.path.join(data_dir, CBIO_STUDY, CBIO_STUDY, "data_clinical_patient.txt")
if not os.path.exists(clin_path):
    clin_path = os.path.join(data_dir, CBIO_STUDY, "data_clinical_patient.txt")

os.makedirs(output_dir, exist_ok=True)
assert os.path.exists(clin_path), f"missing clinical file: {clin_path}"
print(f"Cancer: {CANCER} | study: {CBIO_STUDY} | endpoint: {ENDPOINT}")

Cancer: GBM | study: gbm_tcga | endpoint: OS


In [2]:
# cBioPortal files start with a few lines beginning with '#' (display names, descriptions, types, priority);
# skipping them leaves the column IDs as the header: PATIENT_ID, OS_STATUS, OS_MONTHS, DFS_STATUS, DFS_MONTHS, ...

n_skip = 0

with open(clin_path, encoding="utf-8") as f:
    for line in f:
        if line.startswith("#"):
            n_skip += 1
        else:
            break

# Skipping metadata rows and loading
clinical_raw = pd.read_csv(clin_path, sep="\t", skiprows=n_skip)

# Selecting the columns of the chosen endpoint
status_col, months_col = f"{ENDPOINT}_STATUS", f"{ENDPOINT}_MONTHS"
assert status_col in clinical_raw.columns, f"{status_col} not in this clinical file; available: {[c for c in clinical_raw.columns if c.endswith('_STATUS')]}"
clinical_filtered = clinical_raw[["PATIENT_ID", status_col, months_col]].copy()

# Event counts of every endpoint in the file, for reference
for col in [c for c in clinical_raw.columns if c.endswith("_STATUS")]:
    print(f"{col}: {int(clinical_raw[col].astype(str).str.startswith('1:').sum())} events of {int(clinical_raw[col].notna().sum())} patients")

# The endpoint is saved under the column names OS_STATUS / OS_MONTHS because 04, 05 and 07 read those names
clinical_filtered.columns = ["PATIENT_ID", "OS_STATUS", "OS_MONTHS"]

# Converting status to binary: cBioPortal codes are "0:..." (censored) and "1:..." (event) for every endpoint
clinical_filtered["OS_STATUS"] = clinical_filtered["OS_STATUS"].astype(str).str[:2].map({"0:": 0, "1:": 1})
clinical_filtered["OS_MONTHS"] = pd.to_numeric(clinical_filtered["OS_MONTHS"], errors="coerce")

# Dropping NaN and negative survival times
clinical_clean = clinical_filtered.dropna(axis = 0, how = 'any')
clinical_clean = clinical_clean[clinical_clean["OS_MONTHS"] >= 0]

# Set index:
clinical_clean = clinical_clean.set_index("PATIENT_ID")

print(clinical_filtered.shape, clinical_clean.shape)

clinical_clean.to_csv(os.path.join(output_dir, "clinical_data.csv"))
print(f"Saved clinical_data.csv: {clinical_clean.shape} | events: {clinical_clean['OS_STATUS'].sum()} "
      f"({100 * clinical_clean['OS_STATUS'].mean():.1f}%) | endpoint: {ENDPOINT}")
# Record which endpoint clinical_data.csv holds
with open(os.path.join(output_dir, "clinical_endpoint.txt"), "w") as f:
    f.write(f"{ENDPOINT} from {CBIO_STUDY}, stored in columns OS_STATUS / OS_MONTHS\n")


TUMOR_STATUS: 0 events of 596 patients
OS_STATUS: 492 events of 596 patients
DFS_STATUS: 381 events of 596 patients
(596, 3) (594, 2)
Saved clinical_data.csv: (594, 2) | events: 492.0 (82.8%) | endpoint: OS
